# Shapix Tour

**Elegant runtime shape and dtype checking for NumPy, JAX, and PyTorch arrays.**

Shapix turns array shape annotations into Python objects that [beartype](https://github.com/beartype/beartype) validates at runtime. Dimensions like `N` and `C` are enforced for consistency across all parameters — automatically.

This notebook walks through every major feature.

## 1. Basic usage — shape and dtype checking

Just decorate with `@beartype` and annotate with shapix types. That's it.

In [ ]:
import numpy as np
from beartype import beartype
from shapix import N, C, H, W, Dimension as D
from shapix.numpy import F32


@beartype
def normalize(x: F32[N, C]) -> F32[N, C]:
  """Normalize each row to sum to 1."""
  return x / x.sum(axis=1, keepdims=True)


# This works — correct dtype and shape
result = normalize(np.ones((4, 3), dtype=np.float32))
print("Input shape:  (4, 3)")
print(f"Output shape: {result.shape}")
print(f"Row sums:     {result.sum(axis=1)}")  # each row sums to 1

Input shape:  (4, 3)
Output shape: (4, 3)
Row sums:     [1. 1. 1. 1.]


In [2]:
# Wrong dtype -> caught immediately
try:
  normalize(np.ones((4, 3), dtype=np.float64))
except Exception as e:
  print(f"Wrong dtype: {type(e).__name__}")

# Wrong rank -> caught immediately
try:
  normalize(np.ones((4,), dtype=np.float32))
except Exception as e:
  print(f"Wrong rank:  {type(e).__name__}")

Wrong dtype: BeartypeCallHintParamViolation
Wrong rank:  BeartypeCallHintParamViolation


## 2. Cross-argument dimension consistency

Named dimensions are tracked within a function call. If `N` is bound to 4 by the first parameter, every subsequent parameter must agree.

In [3]:
@beartype
def add(x: F32[N, C], y: F32[N, C]) -> F32[N, C]:
  return x + y


# Same shapes -> OK
result = add(np.ones((4, 3), dtype=np.float32), np.ones((4, 3), dtype=np.float32))
print(f"add((4,3), (4,3)) -> {result.shape}")

# Mismatched N -> caught!
try:
  add(
    np.ones((4, 3), dtype=np.float32),
    np.ones((5, 3), dtype=np.float32),  # N=5 but first arg bound N=4
  )
except Exception as e:
  print(f"add((4,3), (5,3)) -> {type(e).__name__}: N mismatch")

add((4,3), (4,3)) -> (4, 3)
add((4,3), (5,3)) -> BeartypeCallHintParamViolation: N mismatch


Dimensions can be **shared across different positions** in the signature. Here `C` appears as the second dim of `a` and the first dim of `b`, enforcing that the inner dimensions match for matrix multiplication:

In [4]:
@beartype
def matmul(a: F32[N, C], b: F32[C, H]) -> F32[N, H]:
  return a @ b


result = matmul(np.ones((4, 3), dtype=np.float32), np.ones((3, 5), dtype=np.float32))
print(f"matmul((4,3), (3,5)) -> {result.shape}")  # (4, 5)

# Inner dim mismatch -> caught
try:
  matmul(
    np.ones((4, 3), dtype=np.float32),
    np.ones((7, 5), dtype=np.float32),  # C=7, but first arg bound C=3
  )
except Exception:
  print("matmul((4,3), (7,5)) -> C mismatch")

matmul((4,3), (3,5)) -> (4, 5)
matmul((4,3), (7,5)) -> C mismatch


## 3. Sequential calls are independent

Each function invocation gets a fresh set of dimension bindings. Calling the same function twice with different shapes is perfectly fine.

In [5]:
@beartype
def identity(x: F32[N]) -> F32[N]:
  return x


for size in [3, 100, 7, 42, 1]:
  result = identity(np.ones((size,), dtype=np.float32))
  print(f"identity(shape=({size},)) -> {result.shape}")

identity(shape=(3,)) -> (3,)
identity(shape=(100,)) -> (100,)
identity(shape=(7,)) -> (7,)
identity(shape=(42,)) -> (42,)
identity(shape=(1,)) -> (1,)


## 4. Return type checking

The return value is also validated against the spec, including cross-argument consistency with the input dimensions.

In [6]:
@beartype
def buggy_function(x: F32[N, C]) -> F32[N, C]:
  """This function has a bug -- it returns the wrong shape!"""
  return np.zeros((1, 1), dtype=np.float32)  # Oops!


try:
  buggy_function(np.ones((4, 3), dtype=np.float32))
except Exception as e:
  print(f"Return type violation caught: {type(e).__name__}")
  print("  Expected (4, 3) but function returned (1, 1)")

Return type violation caught: BeartypeCallHintReturnViolation
  Expected (4, 3) but function returned (1, 1)


## 5. Fixed dimensions

Use plain integers for dimensions that must match an exact size:

In [ ]:
@beartype
def process_rgb(image: F32[N, D(3), H, W]) -> F32[N, D(3), H, W]:
  """Only accepts images with exactly 3 channels."""
  return image / 255.0


# 3 channels -> OK
result = process_rgb(np.ones((2, 3, 28, 28), dtype=np.float32))
print(f"process_rgb((2, 3, 28, 28)) -> {result.shape}")

# 4 channels -> rejected
try:
  process_rgb(np.ones((2, 4, 28, 28), dtype=np.float32))
except Exception:
  print("process_rgb((2, 4, 28, 28)) -> channel dim must be 3")

process_rgb((2, 3, 28, 28)) -> (2, 3, 28, 28)
process_rgb((2, 4, 28, 28)) -> channel dim must be 3


## 6. Symbolic dimensions (arithmetic)

Dimensions support Python arithmetic. Expressions are evaluated against bound dimension values at runtime.

In [8]:
@beartype
def pad_both_sides(x: F32[N]) -> F32[N + 2]:
  """Pad a 1D array with one zero on each side."""
  return np.pad(x, 1)


result = pad_both_sides(np.ones((5,), dtype=np.float32))
print(f"pad_both_sides(shape=(5,)) -> {result.shape}")  # (7,) = 5+2


@beartype
def flatten(x: F32[N, C]) -> F32[N * C]:
  """Flatten a 2D array into 1D."""
  return x.reshape(-1)


result = flatten(np.ones((3, 4), dtype=np.float32))
print(f"flatten(shape=(3, 4)) -> {result.shape}")  # (12,) = 3*4


# Wrong output shape -> caught
@beartype
def bad_pad(x: F32[N]) -> F32[N + 2]:
  return np.pad(x, 2)  # Pads 2 on each side -> N+4, not N+2!


try:
  bad_pad(np.ones((5,), dtype=np.float32))
except Exception:
  print("bad_pad: expected shape (7,) but got (9,)")

pad_both_sides(shape=(5,)) -> (7,)
flatten(shape=(3, 4)) -> (12,)
bad_pad: expected shape (7,) but got (9,)


## 7. Variadic dimensions

Apply `~` (tilde) to any dimension to make it **variadic** — matching **zero or more** contiguous dimensions. This is perfect for batch dimensions that may or may not be present.

In [9]:
from shapix import B, __


@beartype
def softmax(x: F32[~B, C]) -> F32[~B, C]:
  """Softmax along the last dim, accepting any number of leading batch dims."""
  exp_x = np.exp(x - x.max(axis=-1, keepdims=True))
  return exp_x / exp_x.sum(axis=-1, keepdims=True)


# No batch dim
print(f"softmax((3,))       -> {softmax(np.ones((3,), dtype=np.float32)).shape}")
# One batch dim
print(f"softmax((4, 3))     -> {softmax(np.ones((4, 3), dtype=np.float32)).shape}")
# Two batch dims
print(f"softmax((2, 4, 3))  -> {softmax(np.ones((2, 4, 3), dtype=np.float32)).shape}")

softmax((3,))       -> (3,)
softmax((4, 3))     -> (4, 3)
softmax((2, 4, 3))  -> (2, 4, 3)


Named variadic dims enforce cross-argument consistency on the batch shape. Use `~__` for anonymous variadics when you don't need consistency.

In [10]:
@beartype
def batch_add(x: F32[~B, C], y: F32[~B, C]) -> F32[~B, C]:
  """Both inputs must have the same batch dimensions."""
  return x + y


# Same batch -> OK
result = batch_add(
  np.ones((2, 4, 3), dtype=np.float32), np.ones((2, 4, 3), dtype=np.float32)
)
print(f"batch_add((2,4,3), (2,4,3)) -> {result.shape}")

# Different batch -> caught
try:
  batch_add(
    np.ones((2, 4, 3), dtype=np.float32),
    np.ones((5, 4, 3), dtype=np.float32),  # batch (5,4) != (2,4)
  )
except Exception:
  print("batch_add((2,4,3), (5,4,3)) -> batch mismatch")

batch_add((2,4,3), (2,4,3)) -> (2, 4, 3)
batch_add((2,4,3), (5,4,3)) -> batch mismatch


## 8. Broadcastable dimensions

Apply `+` (unary plus) to any dimension to make it **broadcastable** — size 1 always matches, following NumPy broadcasting rules:

In [11]:
@beartype
def broadcast_add(x: F32[N, C], y: F32[+N, C]) -> F32[N, C]:
  """y's first dim can be 1 (broadcast) or match N."""
  return x + y


# Normal match
result = broadcast_add(
  np.ones((4, 3), dtype=np.float32), np.ones((4, 3), dtype=np.float32)
)
print(f"broadcast_add((4,3), (4,3)) -> {result.shape}")

# Broadcast: +N=1
result = broadcast_add(
  np.ones((4, 3), dtype=np.float32), np.ones((1, 3), dtype=np.float32)
)
print(f"broadcast_add((4,3), (1,3)) -> {result.shape}")

broadcast_add((4,3), (4,3)) -> (4, 3)
broadcast_add((4,3), (1,3)) -> (4, 3)


## 9. Anonymous dimensions

`__` matches any single dimension without binding — useful when you don't care about a particular dimension's consistency.

In [12]:
@beartype
def sum_last_dim(x: F32[__, __, C]) -> F32[__, __]:
  """We only care that the last dim is consistent. First two are anonymous."""
  return x.sum(axis=-1)


result = sum_last_dim(np.ones((4, 3, 5), dtype=np.float32))
print(f"sum_last_dim((4, 3, 5)) -> {result.shape}")

result = sum_last_dim(np.ones((99, 1, 5), dtype=np.float32))
print(f"sum_last_dim((99, 1, 5)) -> {result.shape}  (anonymous dims match anything)")

sum_last_dim((4, 3, 5)) -> (4, 3)
sum_last_dim((99, 1, 5)) -> (99, 1)  (anonymous dims match anything)


## 10. Ellipsis dimensions

`...` (Ellipsis) is a convenient alias for `~__` (anonymous variadic). Use it in array subscripts to match any number of dimensions without binding:

In [ ]:
@beartype
def first_and_last(x: F32[N, ..., C]) -> tuple:
  """Accept arrays with any number of middle dimensions."""
  return (x.shape[0], x.shape[-1])


# 2D — no middle dims
print(
  f"F32[N, ..., C] with (3, 4):       {first_and_last(np.ones((3, 4), dtype=np.float32))}"
)
# 3D — one middle dim
print(
  f"F32[N, ..., C] with (3, 5, 4):    {first_and_last(np.ones((3, 5, 4), dtype=np.float32))}"
)
# 5D — three middle dims
print(
  f"F32[N, ..., C] with (3,2,2,2,4):  {first_and_last(np.ones((3, 2, 2, 2, 4), dtype=np.float32))}"
)

## 10b. Scalar dimension

`Scalar` represents a zero-dimensional array (shape `()`). Use it when a function returns a scalar array:

In [ ]:
from shapix import Scalar


@beartype
def sum_all(x: F32[N]) -> F32[Scalar]:
  """Sum all elements, returning a 0-d array with shape ()."""
  return np.array(x.sum(), dtype=np.float32)  # wrap in 0-d ndarray


a = np.array([1.0, 2.0, 3.0], dtype=np.float32)
result = sum_all(a)
print(f"sum_all({a}) = {result}")
print(f"  shape: {result.shape}  (scalar — zero dimensions)")


# Non-scalar return would be caught
@beartype
def bad_sum(x: F32[N]) -> F32[Scalar]:
  return x  # Returns shape (N,), not ()!


try:
  bad_sum(a)
except Exception:
  print("bad_sum: expected shape () but got (3,)")

## 11. Custom dimensions

Create your own dimension symbols with `Dimension` for domain-specific annotations.

To avoid pyright/Pylance errors, use the `TYPE_CHECKING` pattern:

In [ ]:
import typing as tp
from shapix import Dimension
from shapix.numpy import I64

# Domain-specific dimensions — use TYPE_CHECKING pattern for pyright/Pylance
if tp.TYPE_CHECKING:
  type Vocab = int
  type Embed = int
  type Seq = int
else:
  Vocab = Dimension("Vocab")
  Embed = Dimension("Embed")
  Seq = Dimension("Seq")


@beartype
def embed_tokens(tokens: I64[N, Seq], table: F32[Vocab, Embed]) -> F32[N, Seq, Embed]:
  """Look up token embeddings from an embedding table."""
  return table[tokens]


tokens = np.array([[0, 1, 2], [3, 0, 1]], dtype=np.int64)  # (2, 3)
table = np.random.randn(5, 8).astype(np.float32)  # (5, 8) -> Vocab=5, Embed=8
result = embed_tokens(tokens, table)
print(f"embed_tokens: tokens{tokens.shape} + table{table.shape} -> {result.shape}")
print("  N=2, Seq=3, Vocab=5, Embed=8")

## 12. Nested function calls

Each beartype-decorated function call gets its own dimension memo, so nested calls are fully independent:

In [14]:
@beartype
def inner(x: F32[N]) -> F32[N]:
  """N means something different here than in the outer function."""
  return x * 2


@beartype
def outer(x: F32[N, C]) -> F32[N]:
  """Call inner with just the first column -- different rank!"""
  first_col = x[:, 0]  # shape (N,)
  return inner(first_col)


result = outer(np.ones((4, 3), dtype=np.float32))
print(f"outer((4, 3)) calls inner((4,)) -> {result.shape}")
print("  N=4 in outer's context, N=4 in inner's context (independent memos)")

outer((4, 3)) calls inner((4,)) -> (4,)
  N=4 in outer's context, N=4 in inner's context (independent memos)


## 13. Multiple dtype arrays

Shapix supports all common dtypes and category groups. You can mix different dtypes in one function signature:

In [15]:
from shapix.numpy import F64, I32, Float


@beartype
def weighted_sum(values: F32[N, C], indices: I32[N]) -> F64[C]:
  """Select rows by index and sum, returning float64."""
  selected = values[indices]
  return selected.sum(axis=0).astype(np.float64)


values = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=np.float32)
indices = np.array([0, 2, 0], dtype=np.int32)
result = weighted_sum(values, indices)
print(f"weighted_sum: values{values.shape} + indices{indices.shape} -> {result.shape}")
print(f"  Result dtype: {result.dtype}")


# Float accepts any float dtype
@beartype
def any_float(x: Float[N]) -> Float[N]:
  return x


print(f"\nFloat accepts float32: {any_float(np.ones(3, dtype=np.float32)).dtype}")
print(f"Float accepts float64: {any_float(np.ones(3, dtype=np.float64)).dtype}")
print(f"Float accepts float16: {any_float(np.ones(3, dtype=np.float16)).dtype}")

weighted_sum: values(3, 3) + indices(3,) -> (3,)
  Result dtype: float64

Float accepts float32: float32
Float accepts float64: float64
Float accepts float16: float16


## 14. Custom array types

Use `make_array_type` and `DtypeSpec` to create types for custom array classes or custom dtype combinations:

In [16]:
from shapix import make_array_type, DtypeSpec

# Custom dtype: only float32 or float16 (e.g. for mixed-precision training)
MIXED_PRECISION = DtypeSpec("MixedPrecision", frozenset({"float16", "float32"}))
MixedArray = make_array_type(np.ndarray, MIXED_PRECISION)


@beartype
def mixed_op(x: MixedArray[N, C]) -> MixedArray[N, C]:
  return x


# float32 -> OK
result = mixed_op(np.ones((4, 3), dtype=np.float32))
print(f"MixedArray accepts float32: {result.dtype}")

# float16 -> OK
result = mixed_op(np.ones((4, 3), dtype=np.float16))
print(f"MixedArray accepts float16: {result.dtype}")

# float64 -> rejected
try:
  mixed_op(np.ones((4, 3), dtype=np.float64))
except Exception:
  print("MixedArray rejects float64")

MixedArray accepts float32: float32
MixedArray accepts float16: float16
MixedArray rejects float64


## 14b. Like types (input validation)

`Like` types accept scalars, nested sequences of any depth, or arrays. Use them for function inputs that will be converted. Dtype compatibility uses NumPy's `same_kind` casting rules by default.

Like types **must be subscripted**: use `[...]` for any shape, or `[N, C]` for specific dimensions.

In [ ]:
from shapix.numpy import F32Like


# F32Like[...] accepts any shape — scalars, lists, arrays
@beartype
def to_array(x: F32Like[...]) -> np.ndarray:
  return np.asarray(x, dtype=np.float32)


print("F32Like[...] accepts:")
print(f"  scalar:      {to_array(3.14).shape}")
print(f"  1D list:     {to_array([1.0, 2.0, 3.0]).shape}")
print(f"  2D list:     {to_array([[1, 2], [3, 4]]).shape}")
print(f"  ndarray:     {to_array(np.ones((3, 4))).shape}")
print(
  f"  deep nested: {to_array([[[[[[1.0]]]]]])}.shape = {to_array([[[[[[1.0]]]]]]).shape}"
)


# Like types with shape constraints
@beartype
def process(x: F32Like[N, C]) -> np.ndarray:
  return np.asarray(x, dtype=np.float32)


result = process([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(f"\nF32Like[N, C] with [[1,2,3],[4,5,6]]: shape {result.shape}")

# int32 -> float32 is OK under same_kind casting
print(f"  int array accepted: {process(np.ones((2, 3), dtype=np.int32)).shape}")

# complex -> float32 is NOT OK under same_kind
try:
  process(np.ones((2, 3), dtype=np.complex128))
except Exception:
  print("  complex128 rejected (not same_kind as float32)")

## 14c. ScalarLike types (range-validated scalars)

ScalarLike types validate individual scalar values with range checking. They check the *value*, not the shape — perfect for function parameters that should be a single number within a specific range.

In [ ]:
from shapix.numpy import U8ScalarLike, I8ScalarLike


@beartype
def set_pixel(value: U8ScalarLike) -> int:
  """Accepts int in [0, 255] range (uint8 bounds)."""
  return int(value)


print("U8ScalarLike (uint8 range [0, 255]):")
print(f"  set_pixel(0)   = {set_pixel(0)}")
print(f"  set_pixel(128) = {set_pixel(128)}")
print(f"  set_pixel(255) = {set_pixel(255)}")

try:
  set_pixel(256)
except Exception:
  print("  set_pixel(256) -> rejected (out of uint8 range)")

try:
  set_pixel(-1)
except Exception:
  print("  set_pixel(-1)  -> rejected (negative for unsigned)")


@beartype
def clamp_i8(value: I8ScalarLike) -> int:
  """Accepts int in [-128, 127] range (int8 bounds)."""
  return int(value)


print("\nI8ScalarLike (int8 range [-128, 127]):")
print(f"  clamp_i8(-128) = {clamp_i8(-128)}")
print(f"  clamp_i8(127)  = {clamp_i8(127)}")

try:
  clamp_i8(128)
except Exception:
  print("  clamp_i8(128)  -> rejected (out of int8 range)")

## 14d. Endianness variants

For multi-byte dtypes, shapix provides endianness-constrained variants with `LE` (little-endian), `BE` (big-endian), and `N` (native) suffixes. Single-byte types (`Bool`, `I8`, `U8`) have no variants since byte order is irrelevant for them.

In [ ]:
from shapix.numpy import F32LE, F32N


@beartype
def process_le(x: F32LE[N]) -> F32LE[N]:
  """Only accepts little-endian float32 arrays."""
  return x


# Little-endian -> OK
arr_le = np.ones(5, dtype="<f4")  # explicit little-endian
result = process_le(arr_le)
print(f"F32LE accepts '<f4' (little-endian): shape {result.shape}")

# Big-endian -> rejected
arr_be = np.ones(5, dtype=">f4")  # explicit big-endian
try:
  process_le(arr_be)
except Exception:
  print("F32LE rejects '>f4' (big-endian)")

# Without endianness suffix, F32 accepts both

print("\nF32 (no suffix) accepts '<f4': True")
print("F32 (no suffix) accepts '>f4': True")


# Native endianness
@beartype
def native_only(x: F32N[N]) -> F32N[N]:
  return x


result = native_only(np.ones(5, dtype=np.float32))  # native
print(f"\nF32N accepts native float32: shape {result.shape}")

## 14e. Structured dtypes

Use `Structured()` for NumPy structured (record) dtypes. The exact field names, types, and order are validated:

In [ ]:
from shapix.numpy import Structured

# Define a structured dtype for 2D points
Point = Structured([("x", np.float32), ("y", np.float32)])


@beartype
def centroid(pts: Point[N]) -> np.ndarray:
  """Compute centroid of N points."""
  return np.array([pts["x"].mean(), pts["y"].mean()], dtype=np.float32)


# Correct fields -> OK
points = np.array(
  [(1.0, 2.0), (3.0, 4.0), (5.0, 6.0)], dtype=[("x", "<f4"), ("y", "<f4")]
)
result = centroid(points)
print(f"centroid of {len(points)} points: ({result[0]:.1f}, {result[1]:.1f})")

# Wrong field names -> rejected
wrong_fields = np.array([(1.0, 2.0), (3.0, 4.0)], dtype=[("a", "<f4"), ("b", "<f4")])
try:
  centroid(wrong_fields)
except Exception:
  print("Structured rejects wrong field names (a, b) vs (x, y)")


# Shape checking still works on structured arrays
@beartype
def pair_points(a: Point[N], b: Point[N]) -> None:
  """Both arrays must have the same number of points."""
  pass


pair_points(points, points[:3])  # N=3 for both -> OK
print("pair_points with matching N=3: OK")

try:
  pair_points(points, points[:2])  # N=3 vs N=2
except Exception:
  print("pair_points with N=3 vs N=2: rejected")

## 14f. Custom Like types with `make_array_like_type`

Use `make_array_like_type` to create custom Like type factories. The `casting` parameter controls dtype strictness using NumPy casting rules: `"no"` < `"equiv"` < `"safe"` < `"same_kind"` (default) < `"unsafe"`.

In [ ]:
from shapix import make_array_like_type, DtypeSpec
from shapix._dtypes import FLOAT32

# Default casting='same_kind' — allows int->float but not complex->float
F32LikeDefault = make_array_like_type(FLOAT32, name="F32LikeDefault")


@beartype
def default_cast(x: F32LikeDefault[...]) -> np.ndarray:
  return np.asarray(x, dtype=np.float32)


print("same_kind casting (default):")
print(f"  int list accepted:    {default_cast([1, 2, 3]).dtype}")
print(f"  float list accepted:  {default_cast([1.0, 2.0]).dtype}")

# Strict casting='no' — only exact dtype match
F32Strict = make_array_like_type(FLOAT32, casting="no", name="F32Strict")


@beartype
def strict_cast(x: F32Strict[...]) -> np.ndarray:
  return np.asarray(x, dtype=np.float32)


print("\nstrict casting (casting='no'):")
print(f"  float32 accepted: {strict_cast(np.ones(3, dtype=np.float32)).dtype}")

try:
  strict_cast(np.ones(3, dtype=np.float64))
except Exception:
  print("  float64 rejected (exact match only)")

# Custom dtype combo
MIXED = DtypeSpec("Mixed", frozenset({"float16", "float32"}))
MixedLike = make_array_like_type(MIXED, name="MixedLike")


@beartype
def mixed_input(x: MixedLike[...]) -> np.ndarray:
  return np.asarray(x)


print(f"\nMixedLike accepts float32: {mixed_input(np.ones(3, dtype=np.float32)).dtype}")
print(f"MixedLike accepts float16: {mixed_input(np.ones(3, dtype=np.float16)).dtype}")

## 15. Explicit memo management

For advanced use cases or guaranteed correctness, use `@shapix.check` or `check_context`:

In [ ]:
import shapix
from beartype import BeartypeConf
from beartype.door import is_bearable


# @shapix.check decorator -- explicit memo push/pop around the call
@shapix.check
@beartype
def checked_add(x: F32[N, C], y: F32[N, C]) -> F32[N, C]:
  return x + y


result = checked_add(
  np.ones((4, 3), dtype=np.float32), np.ones((4, 3), dtype=np.float32)
)
print(f"@shapix.check + @beartype: {result.shape}")


# Combined mode: @shapix.check(conf=...) applies @beartype automatically
@shapix.check(conf=BeartypeConf())
def combined_add(x: F32[N, C], y: F32[N, C]) -> F32[N, C]:
  return x + y


result = combined_add(
  np.ones((4, 3), dtype=np.float32), np.ones((4, 3), dtype=np.float32)
)
print(f"@shapix.check(conf=...): {result.shape}  (beartype applied automatically)")

# check_context -- for manual is_bearable() checks with shared dimensions
with shapix.check_context():
  x = np.ones((4, 3), dtype=np.float32)
  y = np.ones((4, 5), dtype=np.float32)

  result1 = is_bearable(x, F32[N, C])
  result2 = is_bearable(y, F32[N, H])  # N=4 from x, H=5 from y

  print("\ncheck_context:")
  print(f"  x(4,3) matches F32[N, C]: {result1}")
  print(f"  y(4,5) matches F32[N, H]: {result2}  (N=4 consistent, H=5 binds)")

# Outside the context, dimensions are fresh
with shapix.check_context():
  z = np.ones((10, 7), dtype=np.float32)
  result3 = is_bearable(z, F32[N, C])
  print(f"  z(10,7) in new context matches F32[N, C]: {result3}  (N=10, fresh memo)")

## 16. Putting it all together — a mini neural network forward pass

Here's a realistic example combining multiple features:

In [18]:
from shapix import B, Dimension

# Domain-specific dimensions
Features = Dimension("Features")
Hidden = Dimension("Hidden")
Classes = Dimension("Classes")


@beartype
def linear(x: F32[B, Features], weight: F32[Features, Hidden]) -> F32[B, Hidden]:
  """A single linear layer: y = x @ W."""
  return x @ weight


@beartype
def relu(x: F32[B, Hidden]) -> F32[B, Hidden]:
  """ReLU activation."""
  return np.maximum(x, 0)


@beartype
def classifier(
  x: F32[B, Features], w1: F32[Features, Hidden], w2: F32[Hidden, Classes]
) -> F32[B, Classes]:
  """Two-layer classifier: linear → relu → linear."""
  h = linear(x, w1)
  h = relu(h)
  # For the second linear layer, we need different dim names
  logits = h @ w2
  return logits


# Set up dimensions
batch, features, hidden, classes = 8, 10, 32, 5
x = np.random.randn(batch, features).astype(np.float32)
w1 = np.random.randn(features, hidden).astype(np.float32)
w2 = np.random.randn(hidden, classes).astype(np.float32)

logits = classifier(x, w1, w2)
print("✓ classifier:")
print(f"  Input:   {x.shape}  (B={batch}, Features={features})")
print(f"  W1:      {w1.shape} (Features={features}, Hidden={hidden})")
print(f"  W2:      {w2.shape} (Hidden={hidden}, Classes={classes})")
print(f"  Output:  {logits.shape}  (B={batch}, Classes={classes})")
print("\n  All dimensions verified: Features matches across x and w1,")
print("  Hidden matches across w1 output and w2 input — automatically!")

✓ classifier:
  Input:   (8, 10)  (B=8, Features=10)
  W1:      (10, 32) (Features=10, Hidden=32)
  W2:      (32, 5) (Hidden=32, Classes=5)
  Output:  (8, 5)  (B=8, Classes=5)

  All dimensions verified: Features matches across x and w1,
  Hidden matches across w1 output and w2 input — automatically!


## 17. Tree annotations

`Tree` annotations validate all leaves in a nested structure (dicts, lists, tuples). Requires `optree` or `jax`.

Import `Tree` from an explicit backend module:
- `from shapix.optree import Tree` — backed by optree
- `from shapix.jax import Tree` — backed by jax.tree_util

In [ ]:
from shapix.optree import Tree
from shapix import T


# Basic leaf checking — all leaves must match
@beartype
def process(data: Tree[F32[N, C]]) -> Tree[F32[N, C]]:
  return data


params = {
  "weights": np.ones((3, 4), dtype=np.float32),
  "bias": np.ones((3, 4), dtype=np.float32),
}
process(params)
print("Tree[F32[N, C]] with dict of (3,4) arrays: OK")

# Wrong dtype in a leaf -> caught
try:
  process({"w": np.ones((3, 4), dtype=np.float64)})
except Exception:
  print("Tree[F32[N, C]] with float64 leaf: rejected")

In [ ]:
# Structure binding — T enforces identical tree shapes across arguments
@shapix.check
@beartype
def add_trees(x: Tree[F32[N], T], y: Tree[F32[N], T]) -> Tree[F32[N]]:
  return x


x_tree = {"a": np.ones(3, dtype=np.float32), "b": np.ones(3, dtype=np.float32)}
y_tree = {"a": np.ones(3, dtype=np.float32), "b": np.ones(3, dtype=np.float32)}
add_trees(x_tree, y_tree)
print("Tree[F32[N], T] with matching dict structures: OK")

# Different structure -> caught
try:
  add_trees({"a": np.ones(3, dtype=np.float32)}, [np.ones(3, dtype=np.float32)])
except Exception:
  print("Tree[F32[N], T] with dict vs list: rejected (different structure)")

In [ ]:
# Multi-level matching
# T, ... = top-level only (subtrees can differ)
# ..., T = bottom-level only (leaf-adjacent containers)
# T, S   = T = top level, S = full remaining structure


@shapix.check
@beartype
def top_only(x: Tree[F32[N], T, ...], y: Tree[F32[N], T, ...]) -> Tree[F32[N]]:
  return x


# Same top-level (dict with keys "a", "b"), different nesting below
x = {"a": [np.ones(3, dtype=np.float32)], "b": np.ones(3, dtype=np.float32)}
y = {"a": np.ones(3, dtype=np.float32), "b": {"c": np.ones(3, dtype=np.float32)}}
top_only(x, y)
print("Tree[F32[N], T, ...] — same top-level, different subtrees: OK")

## Cheat sheet

| Syntax | Meaning | Example |
|--------|---------|---------|
| `N` | Named dim — bind & enforce | `F32[N, C]` |
| `3` | Fixed dim — exact match | `F32[3, N]` |
| `N + 1` | Symbolic — arithmetic expression | `F32[N + 1]` |
| `~B` | Variadic — zero or more dims | `F32[~B, C]` |
| `+N` | Broadcastable — size 1 OK | `F32[+N, C]` |
| `~+B` | Broadcastable variadic | `F32[~+B, C]` |
| `__` | Anonymous — any size, no binding | `F32[__, C]` |
| `~__` | Anonymous variadic — any dims | `F32[~__, C]` |
| `...` | Ellipsis (alias for `~__`) | `F32[N, ..., C]` |
| `Scalar` | Scalar — zero dimensions | `F32[Scalar]` |

**Array types by backend:**
- NumPy: `shapix.numpy.F32`, `I64`, `Shaped`, `V`, `Str`, `DT64`, ...
- JAX: `shapix.jax.F32`, `BF16`, ...
- PyTorch: `shapix.torch.F32`, `BF16`, ...

**Endianness variants:** `F32LE`, `I64BE`, `I32N` (LE/BE/N suffixes)

**Structured dtypes:** `Structured([("x", np.float32), ("y", np.float32)])`

**Like types** (scalar | array | nested sequence):
- `shapix.numpy.F32Like[N, C]`, `IntLike[...]`, `ShapedLike[...]`, ...
- Also available in `shapix.jax` and `shapix.torch`
- Must be subscripted: `F32Like[...]` or `F32Like[N, C]`

**ScalarLike types** (range-validated scalars):
- `I8ScalarLike` ([-128, 127]), `U8ScalarLike` ([0, 255]), `F32ScalarLike`, ...
- `IntScalarLike`, `FloatScalarLike`, `NumScalarLike`, ...
- `StringLike` (str | np.str_)

**Custom types:**
- `make_array_type(array_class, dtype_spec)` — custom array type
- `make_array_like_type(dtype_spec, casting="same_kind", name="...")` — custom Like type
- `DtypeSpec("Name", frozenset({"float32", "float16"}))` — custom dtype

**Tree annotations** (requires `optree` or `jax`):
- `Tree[F32[N, C]]` — leaf type checking
- `Tree[F32[N], T]` — full structure binding
- `Tree[F32[N], T, ...]` — top-level only
- `Tree[F32[N], ..., T]` — bottom-level only
- `Tree[F32[N], T, S]` — multi-level (T = top, S = rest)

**Decorators & context managers:**
- `@shapix.check` — explicit memo management
- `@shapix.check(conf=BeartypeConf())` — combined memo + beartype
- `shapix.check_context()` — for manual `is_bearable()` checks